In [ ]:
try:
    from openmdao.utils.notebook_utils import notebook_mode
except ImportError:
    !python -m pip install openmdao[notebooks]

# The Optimization isn't Working

Getting an optimization to run smoothly and successfully can be a challenge, and when problems arise, the root cause is often not very obvious. This guide is meant to give you some tips for debugging your optimization by identifying certain common problems that can occur and teaching you how to recognize them. 

## Problems with the model



### The solvers aren't converging



### Numerical overflow or underflow

### Missing connections or promotions

### Model structure problems

## Problems with the model's derivatives

If you are using a gradient optimizer, then any problem that introduces innaccuracies into their computation can degrade the performance of the optimizer or cause it to fail. OpenMDAO provides the optimizer the full gradient of the objective and constraints with respect to the design variables whenever it requests it. This tells the optimizer the direction it should explore with a line search. If the gradient is not accurate, the optimizer will have trouble finding a new point that improves over the previous one. 


| Optimizer | Behavior |
| :--- | :--- |
| General | Hits maximum number of iterations |
| SLSQP | Iteration limit reached    (Exit mode 9) |
| SNOPT  | XXX  The objective gradients seem to be incorrect. |
| | SNOPTC EXIT  30 -- resource limit error |
| | SNOPTC INFO  32 -- major iteration limit reached |
| IPOPT | ??? |

### The user-supplied partial derivatives are incorrect

A common cause of gradient innacuracies is mistakes in the partial derivatives defined on one or more components. Before running any optimization, you should check the partial derivatives for all of the `Components` in your model. You can do this by running `check_partials` on your `Problem` and carefully checking the results. For complete details on how to use this feature, see [Verifying Partial Derivatives are Correct](../core_features/working_with_derivatives/main_check_partials.ipynb)

The following example shows how to use `check_partials` to verify the derivatives of a simple model with a component that contains the equation for a paraboloid with two input variables. There is a mistake in one of the declared partials:

In [ ]:
import openmdao.api as om

class Paraboloid(om.ExplicitComponent):

    def setup(self):
        self.add_input('x', val=0.0)
        self.add_input('y', val=0.0)
        self.add_output('f_xy', val=0.0)

    def setup_partials(self):
        self.declare_partials('*', '*')

    def compute(self, inputs, outputs):
        x = inputs['x']
        y = inputs['y']

        outputs['f_xy'] = (x-3.0)**2 + x*y + (y+4.0)**2 - 3.0

    def compute_partials(self, inputs, partials):
        """
        There is an error in one of the derivatives.
        """
        x = inputs['x']
        y = inputs['y']

        partials['f_xy', 'x'] = 2.0*x - 6.0 - y
        partials['f_xy', 'y'] = 2.0*y + 8.0 + x

We wish to find the minimum of this equation by varying the two design variables, so we construct an OpenMDAO model. The optimization is not working, so we check the partials:

In [ ]:
prob = om.Problem()
model = prob.model

model.add_subsystem('parab', Paraboloid(), promotes=['*'])

prob.driver = om.ScipyOptimizeDriver(optimizer='SLSQP')
model.add_objective('f_xy')
model.add_design_var('x', lower=-50, upper=50)
model.add_design_var('y', lower=-50, upper=50)

prob.setup(force_alloc_complex=True)
prob.run_model()

derivs = prob.check_partials(method='cs', compact_print=True)

The partial derivatives look perfect. However, our initial design point (x, y) is (0, 0). This can be degenerate in cases where x and y appear in the expression for the partial derivative. Let's try a different point:

In [ ]:
prob.set_val('x', 4.0)
prob.set_val('y', 3.0)

derivs = prob.check_partials(method='cs', compact_print=True)

Now we can see that the derivative of 'f_xy' with respect to 'x' clearly has a mistake in it. Checking the math reveals a sign error in the "y" term. This highlights an important consideration when testing your component's derivatives. Depending on the structure of the equations, certain combinations of component inputs can cancel out parts of the derivative. A thorough test of the partials may require testing at multiple points or even using randomized inputs. We didn't find the error here until we tried a second point.

The code output below shows the failed optimization that this problem produces. As an exercise, you can fix the mistake and run it again to verify that the optimization is successful.

In [ ]:
prob.run_driver()

### The OpenMDAO-computed total derivatives are incorrect

Even if all of the partial derivatives are correct, you can still have innaccuracies in the computed total derivatives. 

### There are discontinuities or singularties in the derivatives

You may still have problems in your derivatives even after checking that the partial and total derivatives are computed accurately at selected values for your design variables and internal states.

### Debugging more complicated problems

## Problems with the optimization

### The optimizer is driving the model places it shouldn't go

### The problem is poorly scaled

### The convergence criteria are too tight

### Another optimizer might be a better choice

## Is there a bug in OpenMDAO?